# Faruq-v3 — multilevel fusion CM512

Kontrol cache-only: P5 dan P3+P4+P5 sama-sama diproyeksikan menjadi 512 dimensi dengan PCA train-only lalu diuji menggunakan ridge identik. Tidak ada inference atau training detector dan tidak ada rank sweep.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-predicted-roi-transfer-v1/predicted_roi_transfer.json',
    'experiments/faruq-v3-predicted-roi-transfer-v1/predicted_roi_cache_train.npz',
    'experiments/faruq-v3-predicted-roi-transfer-v1/predicted_roi_cache_val.npz',
))
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
TRANSFER = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-predicted-roi-transfer-v1/predicted_roi_transfer.json')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-fusion-cm512-v1/fusion_cm512.json'
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('TRANSFER  :', TRANSFER)
print('OUTPUT    :', OUTPUT)

In [ ]:
from coffee_detector.analysis.faruq_v3_fusion_cm512 import run_faruq_v3_fusion_cm512
result = run_faruq_v3_fusion_cm512(
    CHECKPOINT, TRANSFER, OUTPUT, components=512, ridge=0.01
)
assert result['detector_inference_executed'] is False
assert result['detector_training_executed'] is False
assert result['test_images_accessed'] is False
print('CM512 SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display
rows = []
for name, values in result['results'].items():
    val = values['validation']
    rows.append({
        'representation': name,
        'dimensions': values['dimensions'],
        'train_macro_f1': values['train']['macro_f1'],
        'val_macro_f1': val['macro_f1'],
        'val_balanced_accuracy': val['balanced_accuracy'],
        'val_bottom3_f1': val['bottom3_f1'],
        'val_worst_f1': val['worst_class_f1'],
        'val_top3_accuracy': val['top3_accuracy'],
        'generalization_gap': values['macro_f1_generalization_gap'],
    })
table = pd.DataFrame(rows)
display(table.style.format({column: '{:.2%}' for column in table.columns if column not in ('representation', 'dimensions')}))
fusion = result['results']['P3+P4+P5_CM512']['validation']
bottom = pd.DataFrame(fusion['per_class']).sort_values(['f1', 'class_name']).head(10)
display(bottom.style.format({'f1': '{:.2%}'}))
print('DECISION:', result['decision']['decision'])
print('NEXT:', result['decision']['next_action'])
print('DETAIL:', result['decision'])
print('SUMMARY:', result['summary'])
print('Kirim tabel, bottom-10, dan keputusan. Jangan training detector.')